In [15]:
import pandas as pd

# === 1. 讀取資料 ===
hale = pd.read_csv('WHO_HLE_2021.csv')
life = pd.read_csv('WHO_LE_2021.csv')

# === 2. 取必要欄位並改名 ===
hale = hale[['GEO_NAME_SHORT', 'DIM_TIME', 'DIM_SEX', 'AMOUNT_N']].rename(columns={'AMOUNT_N': 'HALE'})
life = life[['GEO_NAME_SHORT', 'DIM_TIME', 'DIM_SEX', 'AMOUNT_N']].rename(columns={'AMOUNT_N': 'LE'})

# === 3. 合併資料 ===
merged = pd.merge(life, hale, on=['GEO_NAME_SHORT','DIM_TIME','DIM_SEX'], how='inner')

# === 4. 修正 Taiwan 名稱 ===
merged['GEO_NAME_SHORT'] = merged['GEO_NAME_SHORT'].replace({'China, Taiwan Province of China': 'Taiwan'})

# === 5. 計算不健康壽命與比例 ===
merged['ULE'] = merged['LE'] - merged['HALE']
merged['Ratio'] = merged['ULE'] / merged['LE']*100

# === 6. 取最近年份 ===
latest_year = merged['DIM_TIME'].max()
data_latest = merged[merged['DIM_TIME'] == latest_year]

# === 7. 各國平均（不分性別） ===
country_stats = data_latest.groupby('GEO_NAME_SHORT').agg({
    'LE':'mean', 'HALE':'mean', 'ULE':'mean', 'Ratio':'mean'
}).reset_index()

# === 8. 排序 Top10 與 Less10 ===
Top_N = 5
top10 = country_stats.sort_values('Ratio').head(Top_N)
less10 = country_stats.sort_values('Ratio', ascending=False).head(Top_N)

# === 9. 自選國家 ===
selected_countries = ['China', 'Republic of Korea', 'Japan', 'United States of America']
selected = country_stats[country_stats['GEO_NAME_SHORT'].isin(selected_countries)]

# === 10. 輸出 ===
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print(f"=== [{latest_year}] 年 不健康比例最低 前 {Top_N} 個 ===")
print(top10[['GEO_NAME_SHORT','LE','HALE','ULE','Ratio']])

print(f"\n=== [{latest_year}] 年不健康比例最高  前 {Top_N} 個  ===")
print(less10[['GEO_NAME_SHORT','LE','HALE','ULE','Ratio']])

print(f"\n=== [{latest_year}] 年自選國家 ===")
print(selected[['GEO_NAME_SHORT','LE','HALE','ULE','Ratio']])

# === 11. 匯出 CSV（選擇性） ===
top10.to_csv('Top10_UnhealthyRatio.csv', index=False)
less10.to_csv('Less10_UnhealthyRatio.csv', index=False)
selected.to_csv('SelectedCountries_UnhealthyRatio.csv', index=False)


=== [2021] 年 不健康比例最低 前 5 個 ===
                            GEO_NAME_SHORT     LE   HALE   ULE  Ratio
46   Democratic People's Republic of Korea 72.683 64.759 7.923 10.862
81                               Indonesia 68.284 60.677 7.607 11.117
189                               Viet Nam 73.826 65.372 8.454 11.399
95        Lao People's Democratic Republic 68.277 60.435 7.842 11.459
157                        Solomon Islands 64.893 57.412 7.481 11.490

=== [2021] 年不健康比例最高  前 5 個  ===
               GEO_NAME_SHORT     LE   HALE    ULE  Ratio
183  United States of America 76.415 63.933 12.482 16.301
9                   Australia 83.103 70.606 12.496 15.019
99                    Liberia 63.456 53.972  9.484 14.929
97                    Lebanon 74.363 63.255 11.107 14.879
123               New Zealand 82.192 69.976 12.217 14.842

=== [2021] 年自選國家 ===
               GEO_NAME_SHORT     LE   HALE    ULE  Ratio
36                      China 77.698 68.625  9.073 11.646
88                      Japan 